# Test 1 - Iris Species

#### Step 1 - imports

In [ ]:
from sklearn.datasets import load_iris

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
from models import KNNClassifier
from core import Pipeline, GridSearchCV
import metrics as mtr
from preprocessing import train_test_split, normalize, LabelEncoder
import visuals as vis

#### Step 2 - data handling

In [ ]:
iris = load_iris()

X, Y, classes = iris.data, iris.target, iris
X_train, X_test, Y_train, Y_test = train_test_split(X, Y)

In [ ]:
vis.style(style='ticks')
vis.plot_class_distribution(Y, chart='pie', inverse_transform=lambda i: iris.target_names[i])

In [ ]:
pipe = Pipeline([
    ("scaling", normalize.minmax()),
    ("knn", KNNClassifier())
])

grid = GridSearchCV(
    pipe,
    {"knn@k": [1, 3, 5, 10, 20, 50]},
    calculate_metrics=["confusion_matrix"]
)

X_vis = pipe.fit_transform(X_train, Y_train)

#### Step 3 - visualization

In [ ]:

vis.plot_feature_relationships(X_vis, Y_train, feature_labels=iris.feature_names, inverse_transform=lambda i: iris.target_names[i])

#### Step 4 - Training and evaluation

In [ ]:
grid.fit(X_train, Y_train)
for k, acc in zip(grid.cv_results_["param_knn@k"], grid.cv_results_["mean_test_score"]):
    print(f"k = {k}: {round(acc, 4)}")


In [ ]:
probs = grid.predict_proba(X_test)
for i, p in enumerate(mtr.mean_confidence(probs)):
    print(f"{iris.target_names[i]}: {round(p, 4)}")


In [ ]:
vis.plot_confidence_distribution(Y_test, grid.predict(X_test), probs)

In [ ]:
vis.plot_hyperparameter_tuning(grid.cv_results_, "knn@k")

In [ ]:
vis.plot_confusion_matrices(
    grid.cv_results_["metric_confusion_matrix"],
    ["Satosa", "Versicolor", "Virginica"],
    titling=lambda i: f"Confusion Matrix for K ={grid.cv_results_["param_knn@k"][i]}"
)